In [ ]:
import pandas as pd
import numpy as np
import re
import os
import joblib
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. DOWNLOAD & LOCATE DATASET
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# Automatically find any CSV file in the downloaded folder
csv_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded dataset!")

# Pick the first CSV found (usually Resume.csv)
csv_path = csv_files[0]
print(f"✅ Found dataset at: {csv_path}")

df = pd.read_csv(csv_path)

# 2. UPDATED PREPROCESSING (Using Raw Strings to fix SyntaxWarnings)
def clean_resume(resume_text):
    # Use r"" for raw strings to prevent escape sequence errors
    resume_text = re.sub(r'http\S+\s*', ' ', resume_text)
    resume_text = re.sub(r'RT|cc', ' ', resume_text)
    resume_text = re.sub(r'#\S+', '', resume_text)
    resume_text = re.sub(r'@\S+', '  ', resume_text)
    resume_text = re.sub(r'[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', resume_text)
    resume_text = re.sub(r'[^\x00-\x7f]', r' ', resume_text)
    resume_text = re.sub(r'\s+', ' ', resume_text)
    return resume_text.lower().strip()

print("Cleaning data...")
# Identify the correct column for resume text
column_to_use = 'Resume_str' if 'Resume_str' in df.columns else 'Resume'
df['cleaned_resume'] = df[column_to_use].apply(clean_resume)

# 3. LABEL ENCODING
le = LabelEncoder()
df['Category_Encoded'] = le.fit_transform(df['Category'])

# 4. VECTORIZATION (TF-IDF)
tfidf = TfidfVectorizer(sublinear_tf=True, stop_words='english', max_features=2000)
X = tfidf.fit_transform(df['cleaned_resume'])
y = df['Category_Encoded']

# 5. TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. MODEL TRAINING
print("Training the model... this might take a minute.")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 7. EVALUATION
y_pred = model.predict(X_test)
print(f"\n✅ Training Complete! Accuracy: {accuracy_score(y_test, y_pred):.2%}")

# 8. SAVE ASSETS
joblib.dump(model, 'resume_classifier_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("\n--- Files Saved Successfully in Colab ---")

Using Colab cache for faster access to the 'resume-dataset' dataset.
✅ Found dataset at: /kaggle/input/resume-dataset/Resume/Resume.csv
Cleaning data...
Training the model... this might take a minute.

✅ Training Complete! Accuracy: 71.43%

--- Files Saved Successfully in Colab ---


In [1]:
# --- 1. SETUP & INSTALLATION ---
!pip install kagglehub PyPDF2 tqdm -q

import os
import re
import io
import joblib
import pandas as pd
import numpy as np
import kagglehub
import PyPDF2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# Mount Drive to save the final results
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/SmartHirePro/'

# --- 2. DOWNLOAD NEW DATASET ---
print("📥 Downloading 8,900+ PDF dataset...")
path = kagglehub.dataset_download("hadikp/resume-data-pdf")

# --- 3. DATA INGESTION (Turning PDFs into a Table) ---
def clean_text(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text)
    text = re.sub(r'[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

data = []
print("🔍 Extracting text from PDFs (This may take 10-15 minutes)...")

# Folders represent categories
categories = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]

for category in categories:
    cat_folder = os.path.join(path, category)
    files = [f for f in os.listdir(cat_folder) if f.endswith('.pdf')]

    # Using tqdm for a progress bar per category
    for filename in tqdm(files, desc=f"Processing {category}"):
        file_path = os.path.join(cat_folder, filename)
        try:
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() or ""

                if len(text.strip()) > 50: # Ignore empty/short files
                    data.append({'text': clean_text(text), 'category': category})
        except:
            continue

df = pd.DataFrame(data)
print(f"✅ Extraction complete! Total usable resumes: {len(df)}")

# --- 4. MODEL TRAINING ---
print("🧠 Training Random Forest Classifier...")
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

tfidf = TfidfVectorizer(max_features=3000, stop_words='english', sublinear_tf=True)
X = tfidf.fit_transform(df['text'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

# --- 5. EVALUATION & SAVING ---
accuracy = model.score(X_test, y_test)
print(f"📊 Model Accuracy on New Dataset: {accuracy:.2%}")

# Save to your SmartHirePro folder
os.makedirs(SAVE_PATH, exist_ok=True)
joblib.dump(model, os.path.join(SAVE_PATH, 'resume_classifier_model.pkl'))
joblib.dump(tfidf, os.path.join(SAVE_PATH, 'tfidf_vectorizer.pkl'))
joblib.dump(le, os.path.join(SAVE_PATH, 'label_encoder.pkl'))

print(f"💾 All files saved to: {SAVE_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.6 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

In [ ]:
import io
import re
import joblib
import numpy as np
from google.colab import files

# 1. LOAD MODELS (Only run this if they aren't already in memory)
# If you just trained them, they are already in the 'model', 'tfidf', and 'le' variables.
# If not, uncomment these lines:
# model = joblib.load('resume_classifier_model.pkl')
# tfidf = joblib.load('tfidf_vectorizer.pkl')
# le = joblib.load('label_encoder.pkl')

# 2. THE CLEANING FUNCTION (Must be identical to the one used in training)
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'RT|cc', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '  ', text)
    text = re.sub(r'[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# 3. THE PREDICTION LOGIC
def predict_from_pdf(pdf_content):
    try:
        import PyPDF2
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_content))
        raw_text = ""
        for page in pdf_reader.pages:
            raw_text += page.extract_text()

        # Process
        cleaned_text = clean_resume(raw_text)
        vectorized_text = tfidf.transform([cleaned_text])

        # Predict
        prediction_id = model.predict(vectorized_text)[0]
        domain_name = le.inverse_transform([prediction_id])[0]

        # Confidence Score
        probs = model.predict_proba(vectorized_text)
        confidence = np.max(probs) * 100

        print(f"\n✅ Result: {domain_name}")
        print(f"📊 Confidence: {confidence:.2f}%")
        return domain_name

    except Exception as e:
        print(f"❌ Error processing PDF: {e}")
        return None

# 4. THE UPLOAD TRIGGER
print("--- 🚀 Test Your Resume Domain ---")
print("Upload a PDF to see how the model classifies it.")

uploaded = files.upload()

for filename, content in uploaded.items():
    print(f"\nAnalyzing: {filename}...")
    predict_from_pdf(content)

In [ ]:
import io
import re
import joblib
import numpy as np
from google.colab import files

# 1. LOAD MODELS (Only run this if they aren't already in memory)
# If you just trained them, they are already in the 'model', 'tfidf', and 'le' variables.
# If not, uncomment these lines:
# model = joblib.load('resume_classifier_model.pkl')
# tfidf = joblib.load('tfidf_vectorizer.pkl')
# le = joblib.load('label_encoder.pkl')

# 2. THE CLEANING FUNCTION (Must be identical to the one used in training)
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'RT|cc', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '  ', text)
    text = re.sub(r'[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# 3. THE PREDICTION LOGIC
def predict_from_pdf(pdf_content):
    try:
        import PyPDF2
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_content))
        raw_text = ""
        for page in pdf_reader.pages:
            raw_text += page.extract_text()

        # Process
        cleaned_text = clean_resume(raw_text)
        vectorized_text = tfidf.transform([cleaned_text])

        # Predict
        prediction_id = model.predict(vectorized_text)[0]
        domain_name = le.inverse_transform([prediction_id])[0]

        # Confidence Score
        probs = model.predict_proba(vectorized_text)
        confidence = np.max(probs) * 100

        print(f"\n✅ Result: {domain_name}")
        print(f"📊 Confidence: {confidence:.2f}%")
        return domain_name

    except Exception as e:
        print(f"❌ Error processing PDF: {e}")
        return None

# 4. THE UPLOAD TRIGGER
print("--- 🚀 Test Your Resume Domain ---")
print("Upload a PDF to see how the model classifies it.")

uploaded = files.upload()

for filename, content in uploaded.items():
    print(f"\nAnalyzing: {filename}...")
    predict_from_pdf(content)

--- 🚀 Test Your Resume Domain ---
Upload a PDF to see how the model classifies it.


Saving Asjad Khan CV-Dec 2025.pdf to Asjad Khan CV-Dec 2025 (2).pdf

Analyzing: Asjad Khan CV-Dec 2025 (2).pdf...
❌ Error processing PDF: name 'tfidf' is not defined


In [ ]:
import io
import re
import joblib
import numpy as np
from google.colab import files

# 1. LOAD MODELS (Only run this if they aren't already in memory)
# If you just trained them, they are already in the 'model', 'tfidf', and 'le' variables.
# If not, uncomment these lines:
# model = joblib.load('resume_classifier_model.pkl')
# tfidf = joblib.load('tfidf_vectorizer.pkl')
# le = joblib.load('label_encoder.pkl')

# 2. THE CLEANING FUNCTION (Must be identical to the one used in training)
def clean_resume(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'RT|cc', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '  ', text)
    text = re.sub(r'[%s]' % re.escape(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# 3. THE PREDICTION LOGIC
def predict_from_pdf(pdf_content):
    try:
        import PyPDF2
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_content))
        raw_text = ""
        for page in pdf_reader.pages:
            raw_text += page.extract_text()

        # Process
        cleaned_text = clean_resume(raw_text)
        vectorized_text = tfidf.transform([cleaned_text])

        # Predict
        prediction_id = model.predict(vectorized_text)[0]
        domain_name = le.inverse_transform([prediction_id])[0]

        # Confidence Score
        probs = model.predict_proba(vectorized_text)
        confidence = np.max(probs) * 100

        print(f"\n✅ Result: {domain_name}")
        print(f"📊 Confidence: {confidence:.2f}%")
        return domain_name

    except Exception as e:
        print(f"❌ Error processing PDF: {e}")
        return None

# 4. THE UPLOAD TRIGGER
print("--- 🚀 Test Your Resume Domain ---")
print("Upload a PDF to see how the model classifies it.")

uploaded = files.upload()

for filename, content in uploaded.items():
    print(f"\nAnalyzing: {filename}...")
    predict_from_pdf(content)


--- 🚀 Test Your Resume Domain ---
Upload a PDF to see how the model classifies it.


In [ ]:
import joblib
import os

# The exact path from your screenshot
drive_path = '/content/drive/MyDrive/Colab Notebooks/SmartHirePro/'

# Loading the "reanimated" brains
try:
    classifier = joblib.load(os.path.join(drive_path, 'resume_classifier_model.pkl'))
    vectorizer = joblib.load(os.path.join(drive_path, 'tfidf_vectorizer.pkl'))
    encoder = joblib.load(os.path.join(drive_path, 'label_encoder.pkl'))
    print("✅ All models loaded successfully from SmartHirePro folder!")
except FileNotFoundError:
    print("❌ Error: Files not found. Make sure you ran drive.mount('/content/drive') first.")

✅ All models loaded successfully from SmartHirePro folder!


In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import shutil
import os

# This will open a pop-up to authorize access
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Define the destination folder in your Google Drive
drive_path = '/content/drive/MyDrive/ColabNotebooks'

# 2. Create the folder if it doesn't exist
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"Created folder: {drive_path}")

# 3. List of files to copy
files_to_save = ['resume_classifier_model.pkl', 'tfidf_vectorizer.pkl', 'label_encoder.pkl']

# 4. Copy each file
for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(drive_path, f))
        print(f"✅ Successfully saved {f} to Google Drive!")
    else:
        print(f"❌ Error: {f} not found in /content/. Did you run the training code yet?")

print(f"\nYour files are now safe at: My Drive > PrepHire_ML")

Created folder: /content/drive/MyDrive/ColabNotebooks
✅ Successfully saved resume_classifier_model.pkl to Google Drive!
✅ Successfully saved tfidf_vectorizer.pkl to Google Drive!
✅ Successfully saved label_encoder.pkl to Google Drive!

Your files are now safe at: My Drive > PrepHire_ML


In [ ]:
from google.colab import drive
import os
import shutil

# 1. MOUNT DRIVE (To get your saved files)
drive.mount('/content/drive')

# 2. CONFIGURATION
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"
REPO_OWNER = "Komal25252"
REPO_NAME = "PrepHire-app"
BRANCH_NAME = "ml-resume-classification"
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/SmartHirePro"

# 3. GIT IDENTITY
!git config --global user.email "your-github-email@example.com" # Update this
!git config --global user.name "AzamSiddiq"

# 4. PREPARE REPOSITORY
repo_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if os.path.exists(REPO_NAME):
    !rm -rf {REPO_NAME}

print("Cloning repository...")
!git clone {repo_url}
%cd {REPO_NAME}

# Create or switch to your branch
!git checkout -b {BRANCH_NAME} || !git checkout {BRANCH_NAME}

# 5. COPY FILES FROM DRIVE TO REPO
print("Copying models from Google Drive...")
files_to_push = ['resume_classifier_model.pkl', 'tfidf_vectorizer.pkl', 'label_encoder.pkl']

for f in files_to_push:
    source = os.path.join(DRIVE_FOLDER, f)
    if os.path.exists(source):
        shutil.copy(source, f)
        print(f"✅ Ready: {f}")
    else:
        print(f"❌ Error: {f} not found in your Drive folder!")

# 6. COMMIT AND PUSH
print("\nPushing to GitHub...")
!git add .
!git commit -m "feat: uploaded trained resume classification models from Google Drive"
!git push origin {BRANCH_NAME}

print(f"\n✅ SUCCESS! View your files here:")
print(f"https://github.com/{REPO_OWNER}/{REPO_NAME}/tree/{BRANCH_NAME}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning repository...
Cloning into 'PrepHire-app'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 62 (delta 8), reused 61 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 86.66 KiB | 7.22 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/PrepHire-app/PrepHire-app/PrepHire-app/PrepHire-app/PrepHire-app
Switched to a new branch 'ml-resume-classification'
Copying models from Google Drive...
✅ Ready: resume_classifier_model.pkl
✅ Ready: tfidf_vectorizer.pkl
✅ Ready: label_encoder.pkl

Pushing to GitHub...
[ml-resume-classification 3ac33b7] feat: uploaded trained resume classification models from Google Drive
 3 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 label_encoder.pkl
 create mode 100644 resume_classifier_mode